In [0]:
# Databricks notebook source

# COMMAND ----------


In [0]:
from __future__ import annotations

import sys
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.types import StructField, StructType, StringType

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "common").exists():
        repository_root = str(candidate)
        if repository_root not in sys.path:
            sys.path.insert(0, repository_root)
        break

from notebooks.common.audit import add_audit_columns
from notebooks.common.metrics import (
    calculate_data_quality_score,
    create_pipeline_metric_df,
    print_summary,
)
from notebooks.common.paths import PATHS
import importlib
import notebooks.common.audit as audit_module

importlib.reload(audit_module)
add_audit_columns = audit_module.add_audit_columns


In [0]:
NOTEBOOK_VERSION = "1.0.0"
PIPELINE_NAME = "northstar_bronze_to_silver_dependents"

BUSINESS_DATE = "2026-07-01"
WRITE_MODE = "overwrite"

SOURCE_SYSTEM = "Employer HR Systems"
BATCH_ID = "northstar-dependents-20260701"

BRONZE_PATH = (
    f"abfss://bronze@{PATHS.storage_account}.dfs.core.windows.net/"
    "northstar/enrollment/dependents/dependents_20260701.csv"
)

SILVER_PATH = (
    f"abfss://silver@{PATHS.storage_account}.dfs.core.windows.net/"
    "northstar/dependents"
)

QUARANTINE_PATH = (
    f"abfss://silver@{PATHS.storage_account}.dfs.core.windows.net/"
    "northstar/quarantine/dependents"
)

METRICS_PATH = (
    f"abfss://gold@{PATHS.storage_account}.dfs.core.windows.net/"
    "northstar/data_quality_metrics"
)

print(BRONZE_PATH)
print(SILVER_PATH)


In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
)

dependents_schema = StructType([
    StructField("dependent_id", StringType(), True),
    StructField("employee_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("relationship", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("gender", StringType(), True),
    StructField("dependent_status", StringType(), True),
    StructField("_corrupt_record", StringType(), True),
])


def read_bronze_dependents(path: str):

    return (
        spark.read.format("csv")
        .schema(dependents_schema)
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .option("dateFormat", "yyyy-MM-dd")
        .load(path)
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file_path"),
        )
    )


bronze_df = read_bronze_dependents(BRONZE_PATH)

records_read = bronze_df.count()

print(f"Bronze dependent records read: {records_read:,}")

display(bronze_df.limit(10))

In [0]:
# Basic dependent validation

validated_df = (
    bronze_df
    .withColumn(
        "validation_error",
        F.when(
            F.col("dependent_id").isNull(),
            "Missing dependent_id"
        )
        .when(
            F.col("employee_id").isNull(),
            "Missing employee_id"
        )
        .when(
            F.col("relationship").isNull(),
            "Missing relationship"
        )
        .when(
            F.col("date_of_birth").isNull(),
            "Missing date_of_birth"
        )
        .when(
            F.col("_corrupt_record").isNotNull(),
            "Corrupt source record"
        )
    )
)

valid_df = (
    validated_df
    .filter(F.col("validation_error").isNull())
)

quarantine_df = (
    validated_df
    .filter(F.col("validation_error").isNotNull())
)

valid_count = valid_df.count()
quarantine_count = quarantine_df.count()

print(f"Valid rows: {valid_count:,}")
print(f"Rejected rows: {quarantine_count:,}")

display(quarantine_df.limit(10))

In [0]:
silver_df = add_audit_columns(
    valid_df,
    batch_id=BATCH_ID,
    source_system=SOURCE_SYSTEM,
)

quarantine_output_df = add_audit_columns(
    quarantine_df,
    batch_id=BATCH_ID,
    source_system=SOURCE_SYSTEM,
)

display(silver_df.limit(10))

In [0]:
(
    silver_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

(
    quarantine_output_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(QUARANTINE_PATH)
)

silver_written_count = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
    .count()
)

quarantine_written_count = (
    spark.read
    .format("delta")
    .load(QUARANTINE_PATH)
    .count()
)

print(f"Silver rows written: {silver_written_count:,}")
print(f"Quarantine rows written: {quarantine_written_count:,}")

if silver_written_count != valid_count:
    raise RuntimeError(
        "Silver write validation failed: "
        f"expected={valid_count:,}, "
        f"actual={silver_written_count:,}"
    )

if quarantine_written_count != quarantine_count:
    raise RuntimeError(
        "Quarantine write validation failed: "
        f"expected={quarantine_count:,}, "
        f"actual={quarantine_written_count:,}"
    )

In [0]:
data_quality_score = calculate_data_quality_score(
    records_read,
    quarantine_count,
)

print_summary(
    records_read=records_read,
    records_written=silver_written_count,
    rejected_records=quarantine_written_count,
    data_quality_score=data_quality_score,
)

In [0]:
metric_df = create_pipeline_metric_df(
    spark,
    pipeline_name=PIPELINE_NAME,
    notebook_version=NOTEBOOK_VERSION,
    batch_id=BATCH_ID,
    business_date=BUSINESS_DATE,
    source_system=SOURCE_SYSTEM,
    source_path=BRONZE_PATH,
    target_path=SILVER_PATH,
    records_read=records_read,
    records_written=silver_written_count,
    records_rejected=quarantine_written_count,
    run_status="SUCCEEDED",
)

(
    metric_df.write
    .format("delta")
    .mode("append")
    .save(METRICS_PATH)
)

In [0]:
print("\nDependent Bronze-to-Silver processing completed successfully.")
print(f"Notebook version: {NOTEBOOK_VERSION}")
print(f"Silver rows:      {silver_written_count:,}")
print(f"Quarantine rows:  {quarantine_written_count:,}")
print(f"Quality score:    {data_quality_score:.2f}%")
print(f"Metrics path:     {METRICS_PATH}")